Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS Transfer Normalization Deep Dive

**Goal: high cell BA and transfer to entirely held-out people.** Literature review completed before model implementation.14 fixed small models,2seeds,11outer folds=308fits. Current63.2294% predictions reused;no control refits or parity gates. Private GPU research,no competition submission.

In [ ]:
import os,sys,pathlib,io,base64,zipfile,subprocess
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',MPLCONFIGDIR='/tmp/transfer-mpl')
work=pathlib.Path('/tmp/transfer-study');work.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text()))) as z:z.extractall(work)
envdir=pathlib.Path('/tmp/transfer-env')
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(envdir)],check=True)
python=str(envdir/'bin/python')
subprocess.run([sys.executable,'-m','pip','--python',python,'install','--quiet','numpy==2.2.6','scipy==1.15.3','scikit-learn==1.7.2','pandas==2.3.3'],check=True)
out=pathlib.Path('/kaggle/working/transfer-results');out.mkdir(parents=True,exist_ok=True)
print((work/'PROTOCOL.md').read_text(),flush=True)
print('No baseline refits, reproduction gates or preliminary test stage.',flush=True)


## Source evidence and adaptations

In [ ]:
print((work/'literature/REVIEW_RU.md').read_text())

## Run fixed transfer experiments

In [ ]:
with (out/'execution.log').open('w') as log:
    process=subprocess.Popen([python,'-u','run.py','--minutes','600'],cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    code=process.wait()
print('Process exit',code,'saved results',out)
if code:raise RuntimeError('Runner stopped; completed predictions and per-job failures remain in output')


## Complete-fold results

In [ ]:
print((out/'REPORT.md').read_text() if (out/'REPORT.md').exists() else 'See progress.json, execution.log and failure receipts')
print('Learning-curve figures:',len(list(out.rglob('curves.png'))))